# FBSA — Main Training (Colab A100)

Drives the full 30-epoch training. Expected wall-time ~1.5h on A100, ~9 Colab units.

Run **only after** `01_sanity.ipynb` passes.

In [ ]:
import sys
from pathlib import Path
PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))
%pip -q install kagglehub scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/fbsa_runs/v1'

In [ ]:
import kagglehub
data_root = str(Path(kagglehub.dataset_download('briscdataset/brisc2025')) / 'brisc2025' / 'segmentation_task')
print('BRISC at:', data_root)

In [ ]:
from tumor_seg.config import TrainConfig
from tumor_seg.train import main

cfg = TrainConfig(
    data_root=data_root,
    out_dir=OUT_DIR,
    epochs=30,
    batch_size=16,
    lr=3e-4,
    num_slots=2,
)
history = main(cfg)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ep = [r['epoch'] for r in history]
ax[0].plot(ep, [r['train_loss'] for r in history], label='train')
ax[0].plot(ep, [r['val_loss']   for r in history], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(ep, [r['train_dice'] for r in history], label='train')
ax[1].plot(ep, [r['val_dice']   for r in history], label='val')
ax[1].set_title('Dice'); ax[1].legend()
ax[2].plot(ep, [r['val_hd'] for r in history])
ax[2].set_title('Val Hausdorff (px)')
plt.tight_layout(); plt.show()